# Bayesian optimization of group sequential designs

## Problem setup

In Bayesian optimization, the goal is to optimize a blackbox function, $f$. In our case, this function takes inputs of 
$$
D = \{n, u_1, \ell_1, u_2, \ell_2, \cdots, u_k, \ell_k \}
$$

where 

- $D$ is the design (an $n$-tuple set),
- $n$ is the sample size at each analysis point (total for both groups),
- $u_1$ and $\ell_1$ are the upper and lower bounds for the first stage, and
- $k$ is the total number of stages

and outputs several values of interest, $T$ (an $m$-tuple set), including (but not limited to)

- $\alpha$, the type I error
- $\beta$, the type II error
- $\mathbb{E}[N \, | \, \boldsymbol{\delta}]$, the expected sample size $N$ over a range of differences between groups $\boldsymbol{\delta} = \{\delta_1, \delta_2, \dots, \delta_j \}$

Above, $N = k*n$ and 

The expected sample size is calculated across a set of possible true treatment effects $\boldsymbol{\delta} = \{\delta_1, \delta_2, \dots, \delta_j \}$ as a function of the design elements $D$: (1) number of analyses $\{1, 2, \dots, k\}$, (2) number of patients at each analysis $\mathbf{n} = \{n_1, n_2, \dots, n_k\}$, and (3) the upper and lower bounds $\mathbf{u} = (u_1, u_2, \dots, u_k)$ and $\boldsymbol{\ell} = (\ell_1, \ell_2, \dots, \ell_k)$.

It is calculated as
$$
\mathbb{E}[N \, | \, \boldsymbol{\delta}]=\sum_{i=1}^k n_i P(\text{trial stops after analysis }i \, | \, \boldsymbol{\delta})
$$

## Pseudocode for Bayesian optimization loop

In order to generated an optimized clinical trial design, the following steps must be followed:

1. Generate several random trial designs, $\{D_1, D_2, \cdots, D_p\}$.
2. Generate the corresponding outputs, $\{T_1, T_2, \cdots, T_p\}$.
3. Fit a Gaussian process regression model to estimate the blackbox function $f: D \to T$.
4. Perform a step of Bayesian optimization to find the next trial design of interest $D_i$.
5. Obtain the corresponding outputs that correspond to this design $T_i$.
6. Refit the Gaussian process regression model on the new data $n$-tuples $\{(\boldsymbol{D}, \boldsymbol{T}), (D_i, T_i)\}$

Repeat until termination policy is reached.

## Function to minimize

As the above optimization problem applies to clinical trial designs, there are certain feasibility constraints that must be considered. Most importantly, the type I and type II error (or power)&mdash;$\alpha$, $\beta$ (or $1-\beta$),respectively&mdash;must be near the set nominal levels. In other words, if the design requires that a one-sided $\alpha = 0.025$, then feasible designs must have values of $\alpha$ near this value. In Wason et al. (Statist. Med. 2012, 31 301–312), feasible designs are defined as "design[s] for which the significance level and power meet the required constraints."

Though Bayesian optimization can be constrained in such a manner, for simplicity, a penalty term will be included within the objective function $f$, which will take these constraints into account. Again, borrowing from Wason et al., the penalty term is:

$$
\mathcal{L} = \mu \cdot \left( \mathbb{I}_{\{\alpha' > \alpha\}}\cdot\frac{\alpha' - \alpha}{\alpha} + \mathbb{I}_{\{\beta' > \beta\}}\cdot\frac{\beta' - \beta}{\beta}  \right)
$$

where $\mu$ is the sample size for a one-stage design, $\alpha$ and $\beta$ are the set nominal values for type I and II error, respectively, $\alpha'$ and $\beta'$ are the type I and II errors for the new design, and $\mathbb{I}$ is the indicator function.

The function that we aim to minimise, $f$, is the sum of the maximum expected sample size of the design and a penalty function that penalizes designs that are not considered feasible:

$$
f = \max\left\{\mathbb{E}[N \, | \, \boldsymbol{\delta}]\right\} + \mathcal{L}
$$